# **Tech Challenge - Fase 2**
## Pipeline Híbrido para Análise da Alfabetização no Brasil 📚
### Esta parte do código se refere à segmentação em CLOUD para testes antes de subir ao AWS

Ana Caroline Gonçalves Lima, RM - 373735


In [25]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# basedosdados se refere a base que o Governo Brasileiro disponíbiliza para análises
# boto3 envia arquivos para o S3
# pyarrow para salvar em PARQUET

!pip install basedosdados boto3 pyarrow --quiet

In [26]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import basedosdados as bd
import pandas as pd
import boto3
import hashlib
import logging

from datetime import datetime, timezone

In [27]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Configuração dos logs
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)

log = logging.getLogger(__name__)

In [28]:
PROJECT_ID = "tech-challenge-fase-2-502101"

DATASET = "br_inep_avaliacao_alfabetizacao"
TABELA = "uf"

BUCKET = "fiap-alfabetizacao-ana-707472259268-us-east-1-an"

INGESTION_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

CAMADA_BRONZE = "bronze"

In [29]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA BRONZE")
log.info(f"Projeto GCP : {PROJECT_ID}")
log.info(f"Dataset     : {DATASET}")
log.info(f"Tabela      : {TABELA}")
log.info(f"Bucket S3   : {BUCKET}")
log.info("~" * 35)

In [30]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO PARA RECEBER OS DADOS VIA BIGQUERY
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_base_dados(dataset, tabela):
    """
    Lê uma tabela da Base dos Dados utilizando BigQuery.
    """

    log.info(f"Lendo tabela {dataset}.{tabela}")

    query = f"""
    SELECT *
    FROM `basedosdados.{dataset}.{tabela}`
    """

    df_uf = bd.read_sql(
        query=query,
        billing_project_id=PROJECT_ID
    )

    log.info(f"{len(df_uf)} registros encontrados")

    return df_uf

In [31]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# EXIBINDO CABEÇALHO DO DATASET
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_uf = ler_base_dados(DATASET, "uf")

df_uf.head()

Downloading: 100%|██████████|


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,AP,2,3,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,PE,2,5,58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONSTRUINDO A CAMADA BRONZE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def construir_bronze(df, dataset, tabela):

    log.info("Adicionando metadados da camada Bronze")

    df = df.copy()

    df["_ingestion_timestamp"] = INGESTION_TS
    df["_ingestion_date"] = INGESTION_DATE
    df["_source_dataset"] = dataset
    df["_source_table"] = tabela

    df["_record_hash"] = (
        df.astype(str)
          .apply(lambda row: hashlib.md5("".join(row).encode()).hexdigest(), axis=1)
    )

    log.info(f"{len(df)} registros preparados para a Bronze")

    return df

In [33]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# EXIBINDO RETORNO DA CAMADA BRONZE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_bronze = construir_bronze(df_uf, DATASET, "uf")

df_bronze.head()

,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260712_132612,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,2d0977a2867ba140b15c2b278f89f529
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260712_132612,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,99f079ac668f1f828dbbffcc412b2baa
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260712_132612,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,cee4ce9f687a44cbb6161294034f35c5
3,2023,AP,2,3,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260712_132612,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,b56942f3319c32a219282f4802d82442
4,2023,PE,2,5,58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260712_132612,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,a4a3d4435c5cae80794c035fd4945a7e
